In [ ]:
import sys

print(sys.executable)
print(sys.version)

# MuFu Dataset Evaluation

This notebook evaluates candidate bioactivity and toxicity datasets for the MuFu molecular property prediction model.

## 1. Setup

In [ ]:
import pandas as pd

print(pd.__version__)

# remember to add rdkit import later!!

## 2. Bioactivity Dataset Evaluation

### 2.1 MoleculeNet BACE

In [ ]:
bace_df = pd.read_csv("data/bace.csv")

bace_df.head()

In [ ]:
bace_df.shape

In [ ]:
bace_df.columns.tolist()

In [ ]:
bace_df[["mol", "CID", "Class", "Model", "pIC50"]].head()

In [ ]:
important_cols = ["mol", "CID", "Class", "pIC50"] # so we don't keep getting 595 columns

print("Missing value counts:")
print(bace_df[important_cols].isna().sum())

print("\nMissing value percentages:")
print(bace_df[important_cols].isna().mean() * 100)

#### Completeness

- No missing values were found in `mol`, `CID`, `Class`, or `pIC50`.
- Both proposed prediction targets are complete: binary activity (`Class`) and continuous potency (`pIC50`).

In [ ]:
print("Dataset shape:", bace_df.shape)
print("Number of molecules:", len(bace_df))
print("Unique CIDs:", bace_df["CID"].nunique())
print("Unique SMILES:", bace_df["mol"].nunique())

In [ ]:
print("Duplicate CIDs:", bace_df["CID"].duplicated().sum())
print("Duplicate SMILES:", bace_df["mol"].duplicated().sum())

In [ ]:
print("Class counts:")
print(bace_df["Class"].value_counts())

print("\nClass percentages:")
print(bace_df["Class"].value_counts(normalize=True) * 100)

In [ ]:
bace_df["pIC50"].describe()

#### Dataset Size, Uniqueness, and Target Distribution

- The BACE dataset contains **1,513 molecules**. All 1,513 CIDs and raw SMILES strings are unique, with no exact duplicates detected.
- The binary activity target is reasonably balanced: **822 Class 0 compounds (54.3%)** and **691 Class 1 compounds (45.7%)**.
- The continuous `pIC50` target is available for all 1,513 molecules and ranges from approximately **2.54 to 10.52**, with a mean of **6.52** and median of **6.76**.
- Raw uniqueness does not guarantee that all molecules are structurally unique, since different SMILES strings can represent the same molecule. RDKit will be used next to validate and canonicalize the molecular structures.

In [ ]:
from rdkit import Chem

print("RDKit imported successfully")

In [ ]:
# validating SMILES strings; will RDKit interpret all of the SMILES strings in the 
# BACE dataset as valid molecular structures?
bace_df["rdkit_mol"] = bace_df["mol"].apply(Chem.MolFromSmiles)

invalid_count = bace_df["rdkit_mol"].isna().sum()

print("Valid SMILES:", len(bace_df) - invalid_count)
print("Invalid SMILES:", invalid_count)
print("Invalid percentage:", invalid_count / len(bace_df) * 100)

In [ ]:
bace_df["canonical_smiles"] = bace_df["rdkit_mol"].apply(
    lambda mol: Chem.MolToSmiles(mol) if mol is not None else None
)

print("Unique raw SMILES:", bace_df["mol"].nunique())
print("Unique canonical SMILES:", bace_df["canonical_smiles"].nunique())
print("Canonical duplicates:", bace_df["canonical_smiles"].duplicated().sum())

#### Molecular Structure Validity and Canonical Uniqueness

- RDKit successfully parsed **all 1,513 SMILES strings**, with **0 invalid molecular structures** detected.
- Canonicalization produced **1,513 unique canonical SMILES**, confirming that no duplicate molecular structures were detected after accounting for alternative SMILES representations.
- These results indicate high structural validity and uniqueness in the BACE dataset.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold

In [ ]:
bace_df["scaffold"] = bace_df["rdkit_mol"].apply(
    lambda mol: MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
)

unique_scaffolds = bace_df["scaffold"].nunique()

print("Total molecules:", len(bace_df))
print("Unique scaffolds:", unique_scaffolds)
print("Molecules per unique scaffold:", len(bace_df) / unique_scaffolds)

In [ ]:
print("Most common scaffolds:")
print(bace_df["scaffold"].value_counts().head(10))

#### Scaffold Diversity

- The 1,513 BACE molecules contain **671 unique Bemis–Murcko scaffolds**, corresponding to approximately **2.25 molecules per unique scaffold** on average.
- The most common scaffold occurs in **55 molecules (~3.6% of the dataset)**, suggesting that the dataset is not dominated by a single structural framework.
- Overall, the dataset shows substantial scaffold variety, supporting reasonable structural diversity for evaluating generalization across different molecular frameworks.

In [ ]:
print("Unique scaffold percentage:",
      unique_scaffolds / len(bace_df) * 100)

largest_scaffold = bace_df["scaffold"].value_counts().iloc[0]

print("Largest scaffold count:", largest_scaffold)
print("Largest scaffold percentage:",
      largest_scaffold / len(bace_df) * 100)

#### Overall BACE Findings

- BACE contains **1,513 molecules** with complete `pIC50` and binary `Class` prediction targets and no missing critical values.
- The classification target is reasonably balanced (**54.3% Class 0, 45.7% Class 1**), while `pIC50` spans approximately **2.54–10.52**.
- All molecular structures were successfully parsed by RDKit, and no duplicate structures were detected after SMILES canonicalization.
- The dataset contains **671 unique Bemis–Murcko scaffolds (44.3% of molecules)**, with the largest scaffold representing only **3.6%** of molecules, indicating substantial scaffold variety.
- Overall, BACE appears to be a **clean, complete, and structurally varied bioactivity dataset** suitable for both pIC50 regression and binary activity classification. Its main limitation for comparison with ChEMBL is its relatively modest dataset size.

### 2.2 ChEMBL BACE-1

ChEMBL target `CHEMBL4822` corresponds to human Beta-secretase 1 (BACE-1), a single-protein target. ChEMBL reports 27,804 associated activity records prior to filtering for the activity type and data-quality requirements relevant to MuFu.

In [ ]:
import urllib.request
import json

test_url = (
    "https://www.ebi.ac.uk/chembl/api/data/activity.json"
    "?target_chembl_id=CHEMBL4822"
    "&standard_type=IC50"
    "&limit=1"
)

with urllib.request.urlopen(test_url) as response:
    test_data = json.load(response)

print(test_data.keys())
print("Total matching IC50 records:", test_data["page_meta"]["total_count"])

In [ ]:
first_record = test_data["activities"][0]

print(first_record.keys())

In [ ]:
first_record

In [ ]:
base_url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

all_activities = []
offset = 0
limit = 1000

while True:
    url = (
        f"{base_url}"
        f"?target_chembl_id=CHEMBL4822"
        f"&standard_type=IC50"
        f"&limit={limit}"
        f"&offset={offset}"
    )

    with urllib.request.urlopen(url) as response:
        data = json.load(response)

    all_activities.extend(data["activities"])

    print(f"Retrieved {len(all_activities)} records")

    if data["page_meta"]["next"] is None:
        break

    offset += limit

print("\nTotal retrieved:", len(all_activities))

In [ ]:
chembl_bace_df = pd.DataFrame(all_activities)

print("Dataset shape:", chembl_bace_df.shape)

chembl_bace_df.head()

In [ ]:
chembl_bace_df.to_csv(
    "data/chembl_bace1_ic50_raw.csv",
    index=False
)

In [ ]:
quality_cols = [
    "canonical_smiles",
    "molecule_chembl_id",
    "standard_value",
    "standard_units",
    "standard_relation",
    "pchembl_value",
    "potential_duplicate",
    "data_validity_comment",
    "assay_type"
]

print("Missing value counts:")
print(chembl_bace_df[quality_cols].isna().sum())

print("\nMissing value percentages:")
print(chembl_bace_df[quality_cols].isna().mean() * 100)

In [ ]:
print("Standard relations:")
print(chembl_bace_df["standard_relation"].value_counts(dropna=False))

print("\nStandard units:")
print(chembl_bace_df["standard_units"].value_counts(dropna=False))

print("\nPotential duplicate flags:")
print(chembl_bace_df["potential_duplicate"].value_counts(dropna=False))

print("\nData validity comments:")
print(chembl_bace_df["data_validity_comment"].value_counts(dropna=False))

print("\nAssay types:")
print(chembl_bace_df["assay_type"].value_counts(dropna=False))

In [ ]:
print("Total IC50 activity records:", len(chembl_bace_df))
print("Unique molecule ChEMBL IDs:", chembl_bace_df["molecule_chembl_id"].nunique())
print("Unique raw canonical SMILES:", chembl_bace_df["canonical_smiles"].nunique())

#### Raw ChEMBL BACE-1 Data Quality

- The initial query returned **13,777 IC50 activity records**, representing **8,938 unique ChEMBL molecule IDs** and **8,935 unique raw canonical SMILES**.
- Molecular structures are nearly complete, with only **2 records missing canonical SMILES**. Standardized value, unit, and relation information is missing for **161 records (1.17%)**.
- Most reported measurements are exact: **12,913 records use `=`**, while **703 use `<` or `>`** and therefore represent bounded rather than exact IC50 measurements.
- Nearly all standardized IC50 measurements use **nM** units (13,614 records); only 2 use `ug.mL-1`.
- ChEMBL flags **2,126 records as potential duplicates** and **305 records with data-validity concerns** (294 outside the typical range and 11 potential transcription errors).
- Most records are from binding assays (**13,758 B-type records**), with only 19 functional-assay records.
- Compared with MoleculeNet BACE, ChEMBL provides substantially more raw activity data but requires additional quality filtering and consolidation of repeated measurements.

In [ ]:
chembl_clean = chembl_bace_df[
    (chembl_bace_df["standard_relation"] == "=") &
    (chembl_bace_df["standard_value"].notna()) &
    (chembl_bace_df["standard_units"] == "nM") &
    (chembl_bace_df["canonical_smiles"].notna()) &
    (chembl_bace_df["data_validity_comment"].isna()) &
    (chembl_bace_df["potential_duplicate"] == 0) &
    (chembl_bace_df["assay_type"] == "B")
].copy()

print("Raw IC50 records:", len(chembl_bace_df))
print("Records after quality filtering:", len(chembl_clean))
print(
    "Records retained (%):",
    len(chembl_clean) / len(chembl_bace_df) * 100
)

print("Unique molecule IDs:", chembl_clean["molecule_chembl_id"].nunique())
print("Unique SMILES:", chembl_clean["canonical_smiles"].nunique())

#### Quality Filtering

- Applying the initial quality criteria reduced the dataset from **13,777 to 10,631 IC50 activity records**, retaining approximately **77.2%** of the original records.
- The retained records contain **8,117 unique ChEMBL molecule IDs** and **8,116 unique SMILES structures**.
- The filtered ChEMBL data therefore contain substantially more unique molecular structures than MoleculeNet BACE (1,513 molecules), even after quality filtering.
- The number of activity records remains larger than the number of unique molecules, indicating that some compounds have multiple experimental IC50 measurements. These repeated measurements must be evaluated before constructing a molecule-level prediction target.

In [ ]:
measurement_counts = chembl_clean["molecule_chembl_id"].value_counts()

print("Molecules with exactly 1 measurement:",
      (measurement_counts == 1).sum())

print("Molecules with multiple measurements:",
      (measurement_counts > 1).sum())

print("Maximum measurements for one molecule:",
      measurement_counts.max())

print("\nMeasurements per molecule:")
print(measurement_counts.describe())

In [ ]:
chembl_clean["standard_value"] = pd.to_numeric(
    chembl_clean["standard_value"],
    errors="coerce"
)

print("Missing after numeric conversion:",
      chembl_clean["standard_value"].isna().sum())

print("Non-positive IC50 values:",
      (chembl_clean["standard_value"] <= 0).sum())

chembl_clean["standard_value"].describe()

In [ ]:
import numpy as np

chembl_clean["pIC50"] = 9 - np.log10(chembl_clean["standard_value"])

chembl_clean["pIC50"].describe()

In [ ]:
chembl_molecule_df = (
    chembl_clean
    .groupby("molecule_chembl_id", as_index=False)
    .agg({
        "canonical_smiles": "first",
        "pIC50": "median"
    })
)

print("Final molecule-level size:", len(chembl_molecule_df))
chembl_molecule_df.head()

In [ ]:
chembl_important_cols = [
    "molecule_chembl_id",
    "canonical_smiles",
    "pIC50"
]

print("Missing value counts:")
print(chembl_molecule_df[chembl_important_cols].isna().sum())

print("\nMissing value percentages:")
print(chembl_molecule_df[chembl_important_cols].isna().mean() * 100)

In [ ]:
print("Dataset shape:", chembl_molecule_df.shape)
print("Number of molecules:", len(chembl_molecule_df))

print(
    "Unique molecule ChEMBL IDs:",
    chembl_molecule_df["molecule_chembl_id"].nunique()
)

print(
    "Unique SMILES:",
    chembl_molecule_df["canonical_smiles"].nunique()
)

print(
    "Duplicate molecule IDs:",
    chembl_molecule_df["molecule_chembl_id"].duplicated().sum()
)

print(
    "Duplicate SMILES:",
    chembl_molecule_df["canonical_smiles"].duplicated().sum()
)

In [ ]:
chembl_molecule_df["pIC50"].describe()

#### ChEMBL Dataset Summary after Pandas Tests for Completeness, Size, Uniqueness, and Target Distribution

- The cleaned ChEMBL dataset contains **8,117 molecules** with no missing IDs, SMILES, or `pIC50` values.
- All molecule IDs are unique; **8,116 of 8,117 SMILES are unique**, leaving only **1 duplicate structure** to examine with RDKit.
- Molecule-level `pIC50` values range from approximately **2.54 to 10.96**, with a mean of **6.74** and median of **6.84**.
- ChEMBL provides substantially more usable molecules than MoleculeNet BACE (**8,117 vs. 1,513**).

In [ ]:
chembl_molecule_df["rdkit_mol"] = (
    chembl_molecule_df["canonical_smiles"].apply(Chem.MolFromSmiles)
)

invalid_count = chembl_molecule_df["rdkit_mol"].isna().sum()

print("Valid SMILES:", len(chembl_molecule_df) - invalid_count)
print("Invalid SMILES:", invalid_count)
print(
    "Invalid percentage:",
    invalid_count / len(chembl_molecule_df) * 100
)

In [ ]:
chembl_molecule_df["rdkit_canonical_smiles"] = (
    chembl_molecule_df["rdkit_mol"].apply(
        lambda mol: Chem.MolToSmiles(mol) if mol is not None else None
    )
)

print(
    "Unique original SMILES:",
    chembl_molecule_df["canonical_smiles"].nunique()
)

print(
    "Unique RDKit canonical SMILES:",
    chembl_molecule_df["rdkit_canonical_smiles"].nunique()
)

print(
    "RDKit canonical duplicates:",
    chembl_molecule_df["rdkit_canonical_smiles"].duplicated().sum()
)

In [ ]:
chembl_molecule_df["scaffold"] = (
    chembl_molecule_df["rdkit_mol"].apply(
        lambda mol: MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    )
)

chembl_unique_scaffolds = chembl_molecule_df["scaffold"].nunique()

print("Total molecules:", len(chembl_molecule_df))
print("Unique scaffolds:", chembl_unique_scaffolds)

print(
    "Molecules per unique scaffold:",
    len(chembl_molecule_df) / chembl_unique_scaffolds
)

In [ ]:
print("Most common scaffolds:")
print(
    chembl_molecule_df["scaffold"]
    .value_counts()
    .head(10)
)

chembl_largest_scaffold = (
    chembl_molecule_df["scaffold"]
    .value_counts()
    .iloc[0]
)

print(
    "\nUnique scaffold percentage:",
    chembl_unique_scaffolds / len(chembl_molecule_df) * 100
)

print("Largest scaffold count:", chembl_largest_scaffold)

print(
    "Largest scaffold percentage:",
    chembl_largest_scaffold / len(chembl_molecule_df) * 100
)

#### ChEMBL Dataset Summary After RDKit Tests for Molecular Structure Validity and Scaffold Diversity

- RDKit successfully parsed **all 8,117 molecular structures**, with **0 invalid SMILES**.
- RDKit canonicalization identified **8,116 unique structures**, leaving only **1 duplicate structure**.
- The dataset contains **2,877 unique Bemis–Murcko scaffolds**, representing **35.4% of molecules**, with approximately **2.82 molecules per scaffold**.
- The largest scaffold contains **166 molecules (2.05%)**, indicating that the dataset is not dominated by a single scaffold.

### 2.3 Bioactivity Dataset Comparison

#### MoleculeNet BACE vs. ChEMBL BACE-1

Both datasets show strong molecular validity and structural diversity, but they differ substantially in size, preparation, and available prediction targets.

| Criterion | MoleculeNet BACE | ChEMBL BACE-1 |
|---|---:|---:|
| Molecule-level size | 1,513 | 8,117 |
| Missing critical values | 0 | 0 after filtering |
| RDKit-valid structures | 100% | 100% |
| Unique canonical structures | 1,513 | 8,116 |
| Unique scaffolds | 671 | 2,877 |
| Unique scaffold percentage | 44.3% | 35.4% |
| Largest scaffold percentage | 3.6% | 2.1% |
| pIC50 range | 2.54–10.52 | 2.54–10.96 |
| Binary activity label provided | Yes (`Class`) | No |
| Preprocessing required | Low | Higher |

**ChEMBL preprocessing:** The initial ChEMBL query returned 13,777 human BACE-1 IC50 activity records. Records were retained only when they had an exact IC50 measurement (`=`), a numeric standardized value in nM, a molecular structure, no ChEMBL data-validity warning, no potential-duplicate flag, and a binding-assay (`B`) classification. This left 10,631 activity records. Because some molecules had repeated measurements, median pIC50 was used for this evaluation to obtain one value per ChEMBL molecule, resulting in 8,117 molecule-level observations.

**Key comparison:** MoleculeNet BACE is smaller but already curated and immediately provides both `pIC50` and binary `Class` targets. ChEMBL BACE-1 provides substantially more usable molecular data and more unique scaffolds, but requires additional filtering, consolidation of repeated measurements, and definition of a binary activity threshold if classification is required.

## 3. Toxicity Dataset Evaluation

### 3.1 MoleculeNet Tox21

In [ ]:
tox21_df = pd.read_csv("data/tox21.csv.gz")

tox21_df.head()

In [ ]:
print("Dataset shape:", tox21_df.shape)
print("\nColumns:")
print(tox21_df.columns.tolist())

In [ ]:
# define targets
tox21_targets = [
    "NR-AR",
    "NR-AR-LBD",
    "NR-AhR",
    "NR-Aromatase",
    "NR-ER",
    "NR-ER-LBD",
    "NR-PPAR-gamma",
    "SR-ARE",
    "SR-ATAD5",
    "SR-HSE",
    "SR-MMP",
    "SR-p53"
]

In [ ]:
missing_counts = tox21_df[tox21_targets].isna().sum()
missing_percentages = tox21_df[tox21_targets].isna().mean() * 100

completeness = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_percentages,
    "Available Count": len(tox21_df) - missing_counts
})

completeness

In [ ]:
class_balance = []

for target in tox21_targets:
    labeled = tox21_df[target].dropna()

    class_balance.append({
        "Target": target,
        "Class 0": (labeled == 0).sum(),
        "Class 1": (labeled == 1).sum(),
        "Class 0 %": (labeled == 0).mean() * 100,
        "Class 1 %": (labeled == 1).mean() * 100
    })

class_balance_df = pd.DataFrame(class_balance)

class_balance_df

In [ ]:
print("Number of rows:", len(tox21_df))
print("Unique molecule IDs:", tox21_df["mol_id"].nunique())
print("Unique raw SMILES:", tox21_df["smiles"].nunique())

print("Missing molecule IDs:", tox21_df["mol_id"].isna().sum())
print("Missing SMILES:", tox21_df["smiles"].isna().sum())

print("Duplicate molecule IDs:", tox21_df["mol_id"].duplicated().sum())
print("Duplicate raw SMILES:", tox21_df["smiles"].duplicated().sum())

#### Tox21 Dataset Summary After Pandas Tests

- Tox21 contains **7,831 unique molecules**, with no missing or duplicate molecule IDs or SMILES.
- The 12 toxicity targets have **7.2%–25.8% missing labels**, depending on the endpoint.
- All 12 targets are **class imbalanced**, with positive (`Class 1`) rates ranging from approximately **2.9% to 16.2%**.
- Missing labels and class imbalance are therefore important considerations when using Tox21 for toxicity prediction.

In [ ]:
tox21_df["rdkit_mol"] = tox21_df["smiles"].apply(Chem.MolFromSmiles)

invalid_count = tox21_df["rdkit_mol"].isna().sum()

print("Valid SMILES:", len(tox21_df) - invalid_count)
print("Invalid SMILES:", invalid_count)
print("Invalid percentage:", invalid_count / len(tox21_df) * 100)

In [ ]:
tox21_df["canonical_smiles"] = tox21_df["rdkit_mol"].apply(
    lambda mol: Chem.MolToSmiles(mol) if mol is not None else None
)

print("Unique raw SMILES:", tox21_df["smiles"].nunique())
print("Unique canonical SMILES:", tox21_df["canonical_smiles"].nunique())
print("Canonical duplicates:", tox21_df["canonical_smiles"].duplicated().sum())

In [ ]:
tox21_df["scaffold"] = tox21_df["rdkit_mol"].apply(
    lambda mol: MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    if mol is not None else None
)

tox21_unique_scaffolds = tox21_df["scaffold"].nunique()

print("Total molecules:", len(tox21_df))
print("Unique scaffolds:", tox21_unique_scaffolds)

print(
    "Molecules per unique scaffold:",
    len(tox21_df) / tox21_unique_scaffolds
)

In [ ]:
print("Most common scaffolds:")
print(tox21_df["scaffold"].value_counts().head(10))

tox21_largest_scaffold = tox21_df["scaffold"].value_counts().iloc[0]

print(
    "\nUnique scaffold percentage:",
    tox21_unique_scaffolds / len(tox21_df) * 100
)

print("Largest scaffold count:", tox21_largest_scaffold)

print(
    "Largest scaffold percentage:",
    tox21_largest_scaffold / len(tox21_df) * 100
)

#### Tox21 Dataset Summary for RDKit Tests on Molecular Structure and Scaffold 

- RDKit successfully parsed **7,823 of 7,831 structures (99.9%)**; only **8 SMILES were invalid**.
- RDKit canonicalization produced **7,823 unique structures among the 7,823 valid molecules**, with no duplicate valid structures detected.- Tox21 contains **2,325 unique Bemis–Murcko scaffolds**, representing approximately **29.7% of all molecules**.
- Overall, Tox21 contains substantial structural variety, although a notable number of molecules do not produce a non-empty Bemis–Murcko scaffold.

### 3.2 MoleculeNet ClinTox

ClinTox is evaluated as the second candidate toxicity dataset. The analysis examines dataset size, target completeness and class balance, molecular uniqueness, structural validity, and scaffold diversity.

In [ ]:
clintox_df = pd.read_csv("data/clintox.csv.gz")

clintox_df.head()

In [ ]:
print("Dataset shape:", clintox_df.shape)

print("\nColumns:")
print(clintox_df.columns.tolist())

In [ ]:
clintox_targets = ["FDA_APPROVED", "CT_TOX"]

missing_counts = clintox_df[clintox_targets].isna().sum()
missing_percentages = clintox_df[clintox_targets].isna().mean() * 100

clintox_completeness = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing %": missing_percentages,
    "Available Count": len(clintox_df) - missing_counts
})

clintox_completeness

In [ ]:
clintox_class_balance = []

for target in clintox_targets:
    labeled = clintox_df[target].dropna()

    clintox_class_balance.append({
        "Target": target,
        "Class 0": (labeled == 0).sum(),
        "Class 1": (labeled == 1).sum(),
        "Class 0 %": (labeled == 0).mean() * 100,
        "Class 1 %": (labeled == 1).mean() * 100
    })

clintox_class_balance_df = pd.DataFrame(clintox_class_balance)

clintox_class_balance_df

In [ ]:
print("Number of rows:", len(clintox_df))
print("Unique raw SMILES:", clintox_df["smiles"].nunique())

print("Missing SMILES:", clintox_df["smiles"].isna().sum())
print("Duplicate raw SMILES:", clintox_df["smiles"].duplicated().sum())

#### ClinTox Dataset Summary after Pandas Tests

- ClinTox contains **1,484 unique molecules** with no missing or duplicate raw SMILES.
- Both `CT_TOX` and `FDA_APPROVED` are **100% complete**, with no missing labels.
- Both targets are strongly class imbalanced: `CT_TOX` has only **7.5% Class 1**, while `FDA_APPROVED` has **93.7% Class 1**.
- ClinTox is therefore complete and clean, but relatively small and highly imbalanced.

In [ ]:
clintox_df["rdkit_mol"] = clintox_df["smiles"].apply(Chem.MolFromSmiles)

invalid_count = clintox_df["rdkit_mol"].isna().sum()

print("Valid SMILES:", len(clintox_df) - invalid_count)
print("Invalid SMILES:", invalid_count)
print("Invalid percentage:", invalid_count / len(clintox_df) * 100)

In [ ]:
clintox_df["canonical_smiles"] = clintox_df["rdkit_mol"].apply(
    lambda mol: Chem.MolToSmiles(mol) if mol is not None else None
)

print("Unique raw SMILES:", clintox_df["smiles"].nunique())
print("Unique canonical SMILES:", clintox_df["canonical_smiles"].nunique())
print("Canonical duplicates:", clintox_df["canonical_smiles"].duplicated().sum())

In [ ]:
clintox_df["scaffold"] = clintox_df["rdkit_mol"].apply(
    lambda mol: MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    if mol is not None else None
)

clintox_unique_scaffolds = clintox_df["scaffold"].nunique()

print("Total molecules:", len(clintox_df))
print("Unique scaffolds:", clintox_unique_scaffolds)
print(
    "Molecules per unique scaffold:",
    len(clintox_df) / clintox_unique_scaffolds
)


In [ ]:
print("Most common scaffolds:")
print(clintox_df["scaffold"].value_counts().head(10))

clintox_largest_scaffold = clintox_df["scaffold"].value_counts().iloc[0]

print(
    "\nUnique scaffold percentage:",
    clintox_unique_scaffolds / len(clintox_df) * 100
)

print("Largest scaffold count:", clintox_largest_scaffold)

print(
    "Largest scaffold percentage:",
    clintox_largest_scaffold / len(clintox_df) * 100
)

#### Clintox Dataset Summary for RDKit Tests on Molecular Structure and Scaffold

- RDKit successfully parsed **1,480 of 1,484 structures (99.7%)**; only **4 SMILES were invalid**.
- Among the valid molecules, RDKit canonicalization produced **1,461 unique structures**, indicating **19 duplicate valid structures** after canonicalization.
- ClinTox contains **784 unique Bemis–Murcko scaffolds**, representing approximately **52.8% of all molecules**.
- The most common non-empty scaffold occurs in **145 molecules (9.8%)**, indicating substantial structural variety.

### 3.3 Toxicity Dataset Comparison

#### Tox21 vs. ClinTox

| Criterion | Tox21 | ClinTox |
|---|---:|---:|
| Dataset size | 7,831 | 1,484 |
| Prediction targets | 12 toxicity endpoints | 2 targets |
| Missing target labels | 7.2%–25.8% | 0% |
| Positive class rate | 2.9%–16.2% | CT_TOX: 7.5% |
| RDKit-valid structures | 99.9% | 99.7% |
| Unique valid canonical structures | 7,823 | 1,461 |
| Unique scaffolds | 2,325 | 784 |
| Unique scaffold percentage | 29.7% | 52.8% |
| Largest non-empty scaffold | ~18.8% | ~9.8% |

**Key comparison:** Tox21 is substantially larger and provides **12 toxicity-related prediction endpoints**, but contains missing labels and strong class imbalance across the tasks. ClinTox is smaller and also strongly imbalanced, but both of its target columns are complete. Both datasets have high molecular-structure validity and substantial scaffold diversity.

Tox21 therefore offers broader toxicity coverage and substantially more molecular data, while ClinTox provides a smaller, more focused dataset with complete labels. The final choice should consider whether MuFu requires broad multi-endpoint toxicity prediction or a narrower clinical-toxicity prediction task.

#### Bioactivity Classification Threshold

For ChEMBL BACE-1, `pIC50` is the main continuous prediction target. To also create a binary active/inactive label, we need to choose a cutoff.

A cutoff of **pIC50 ≥ 6** is a commonly used default in molecular bioactivity work and corresponds to **IC50 ≤ 1 µM**. A stricter cutoff of **pIC50 ≥ 7** corresponds to **IC50 ≤ 100 nM**.

We compare both cutoffs below just to see how they affect class balance. The goal is not to choose whichever one gives the most balanced classes, but to make sure the selected cutoff is scientifically reasonable and practical for the dataset.

In [ ]:
chembl_molecule_df["Class"] = (
    chembl_molecule_df["pIC50"] >= 6
).astype(int)

print("Class counts:")
print(chembl_molecule_df["Class"].value_counts())

print("\nClass percentages:")
print(
    chembl_molecule_df["Class"].value_counts(normalize=True) * 100
)

In [ ]:
for cutoff in [6, 7]:
    active = (chembl_molecule_df["pIC50"] >= cutoff).sum()
    inactive = (chembl_molecule_df["pIC50"] < cutoff).sum()

    print(f"\npIC50 cutoff = {cutoff}")
    print("Active:", active, f"({active / len(chembl_molecule_df) * 100:.1f}%)")
    print("Inactive:", inactive, f"({inactive / len(chembl_molecule_df) * 100:.1f}%)")

#### Threshold Comparison

- Using **pIC50 ≥ 6**, the dataset contains **5,758 active molecules (70.9%)** and **2,359 inactive molecules (29.1%)**.
- Using the stricter **pIC50 ≥ 7** cutoff produces a more balanced split: **3,694 active (45.5%)** and **4,423 inactive (54.5%)**.
- Although the pIC50 = 7 cutoff gives better class balance, class balance alone is not sufficient justification for changing the biological definition of activity.
- Therefore, **pIC50 ≥ 6 remains the provisional classification threshold**, while pIC50 remains the continuous regression target.

### Toxicity Prediction Targets

MuFu uses toxicity as a multi-task binary classification problem. The planned toxicity head includes **14 outputs total**:

- **12 Tox21 endpoints**
- **2 ClinTox targets:** `CT_TOX` and `FDA_APPROVED`

No new toxicity labels or thresholds need to be created because these datasets already provide binary targets. Tox21 contains missing labels for some molecule-endpoint combinations, so these should be handled using task masking during training rather than treating missing values as negative labels.

Both datasets also show substantial class imbalance, which should be considered during preprocessing and model training. `FDA_APPROVED` should be interpreted as an approval-status outcome rather than a direct toxicity assay.